# 03. Сравнение моделей машинного обучения

Цель ноутбука — показать воспроизводимый процесс сравнения нескольких регрессионных моделей для прогнозирования кассовой выручки фильма. Финальной моделью проекта выбрана XGBoost, но для магистерской важно показать сравнительный анализ.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml_pipeline.feature_engineering import MovieFeatureEngineer

DATASET_PATH = PROJECT_ROOT / "data" / "output.csv"
df = pd.read_csv(DATASET_PATH)
df.shape

## Подготовка данных

Используется та же логика, что и в production-скрипте: строки без положительного `gross` и `budget` удаляются, `gross` является целевой переменной, а входные признаки не содержат `gross`.

In [ ]:
TARGET_COLUMN = "gross"
MODEL_INPUT_COLUMNS = [
    "rating", "genre", "year", "score", "votes", "director", "writer",
    "star", "country", "budget", "company", "runtime",
]

clean_df = df.dropna(subset=["gross", "budget"]).copy()
clean_df["gross"] = pd.to_numeric(clean_df["gross"], errors="coerce")
clean_df["budget"] = pd.to_numeric(clean_df["budget"], errors="coerce")
clean_df = clean_df.dropna(subset=["gross", "budget"])
clean_df = clean_df[(clean_df["gross"] > 0) & (clean_df["budget"] > 0)].copy()

X = clean_df[MODEL_INPUT_COLUMNS]
y = clean_df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

## Общий pipeline

Для честного сравнения все модели используют одинаковую подготовку данных:

1. `MovieFeatureEngineer`;
2. `ColumnTransformer`;
3. `SimpleImputer` для числовых и категориальных признаков;
4. `OneHotEncoder(handle_unknown='ignore')` для категорий;
5. `TransformedTargetRegressor` с `log1p` и `expm1`.

In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=5)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore")


def build_pipeline(regressor):
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", make_one_hot_encoder()),
    ])
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])
    preprocessor = ColumnTransformer([
        ("categorical", categorical_pipeline, MovieFeatureEngineer.categorical_features),
        ("numeric", numeric_pipeline, MovieFeatureEngineer.numeric_features),
    ])
    target_regressor = TransformedTargetRegressor(
        regressor=regressor,
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )
    return Pipeline([
        ("feature_engineering", MovieFeatureEngineer()),
        ("preprocessor", preprocessor),
        ("model", target_regressor),
    ])


def calculate_metrics(y_true, y_pred):
    clipped = np.clip(y_pred, 0, None)
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "r2": r2_score(y_true, y_pred),
        "msle": mean_squared_log_error(y_true, clipped),
        "log_r2": r2_score(np.log1p(y_true), np.log1p(clipped)),
    }

## Обучение и оценка моделей

В ячейке ниже сравниваются линейная регрессия, случайный лес, градиентный бустинг и XGBoost. Если нужно ускорить запуск на слабом компьютере, можно временно закомментировать RandomForest или GradientBoosting.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=120, max_depth=12, min_samples_split=10,
        min_samples_leaf=5, random_state=42, n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=220, learning_rate=0.05, max_depth=4, random_state=42,
    ),
    "XGBoost": XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        objective="reg:squarederror",
    ),
}

results = []
trained_pipelines = {}

for model_name, regressor in models.items():
    print(f"Обучение: {model_name}")
    pipeline = build_pipeline(regressor)
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    metrics = calculate_metrics(y_test, y_pred)
    metrics["model"] = model_name
    results.append(metrics)
    trained_pipelines[model_name] = pipeline

results_df = pd.DataFrame(results).set_index("model").sort_values("r2", ascending=False)
results_df

In [ ]:
ax = results_df[["r2", "log_r2"]].plot(kind="bar", figsize=(10, 5), color=["#ff8a5b", "#2ec4b6"])
ax.set_title("Сравнение качества моделей")
ax.set_ylabel("R2")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## Визуальная проверка лучшей модели

Построим график `actual vs predicted` для модели с лучшим `R2`.

In [ ]:
best_model_name = results_df.index[0]
best_pipeline = trained_pipelines[best_model_name]
best_pred = best_pipeline.predict(X_test)

plt.figure(figsize=(7, 7))
plt.scatter(y_test, best_pred, alpha=0.45, color="#3c73ff")
limit = max(y_test.max(), best_pred.max())
plt.plot([0, limit], [0, limit], color="#ff8a5b", linestyle="--")
plt.title(f"Actual vs Predicted: {best_model_name}")
plt.xlabel("Фактическая касса, USD")
plt.ylabel("Прогноз кассы, USD")
plt.tight_layout()
plt.show()

## Вывод

Для production-слоя проекта используется XGBoost, так как по результатам сравнительного анализа он показывает наилучшее качество прогнозирования на тестовой выборке. В итоговом веб-сервисе сохраняется единый sklearn Pipeline, внутри которого есть feature engineering, preprocessing, модель и обратное преобразование `expm1`.

## Дополнение: связь с историческими скриптами проекта

В папке `archive/legacy_models/models/` находятся ранние эксперименты проекта: линейная регрессия, дерево решений, случайный лес, Gradient Boosting и XGBoost. В production-версии логика была перенесена в единый sklearn Pipeline, чтобы модель можно было безопасно подключить к backend.

In [ ]:
models_dir = PROJECT_ROOT / "models"
model_files = sorted(path.name for path in models_dir.glob("*.py"))
pd.DataFrame({"model_script": model_files})

## Дополнение: почему выбран XGBoost

XGBoost выбран как финальная модель интеллектуального модуля веб-сервиса, потому что по результатам сравнительного анализа он показал лучшее качество на тестовой выборке. В production-слое он используется внутри `TransformedTargetRegressor`, поэтому внешний `predict()` возвращает обычную выручку.

In [ ]:
model_info_path = PROJECT_ROOT / "artifacts" / "model_info.json"
metrics_path = PROJECT_ROOT / "artifacts" / "metrics.json"

import json
with model_info_path.open("r", encoding="utf-8") as file:
    model_info = json.load(file)
with metrics_path.open("r", encoding="utf-8") as file:
    production_metrics = json.load(file)

display(pd.Series(model_info, name="model_info").to_frame())
display(pd.Series(production_metrics, name="production_metrics").to_frame())